# **04 - Conformal Prediction Application**

## **Summary**

---
## **Table of Contents**

---
## **Objective**

## <a id ='data-info'></a> **Dataset Info**
* **Source:** UCI Machine Learning Repository
* **Title:** Appliances Energy Prediction
* **Time Range:** 11th January 2016 - 27th May 2016
* **Primary Target Variable:** `Appliances` (Watt-hour consumed)

## **Initial Setup**

In [ ]:
import pandas as pd
from pathlib import Path
import sys
from sklearn.model_selection import train_test_split
import joblib
import numpy as np
from aci.tracker import ACITracker
import matplotlib.pyplot as plt
import numpy as np

# Add the root directory (one level up from notebooks/) to Python's path
root_dir = Path.cwd().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

from src.preprocess import preprocess_energy, feature_engineering, xgb_preprocess

In [ ]:
PROJECT_ROOT = Path.cwd().parent

# Import csv file
energy = pd.read_csv(f'{PROJECT_ROOT}/data/raw/energydata_complete.csv')

In [ ]:
# Data preprocessing
energy = preprocess_energy(energy)

# Form feature through feature engineering
energy = feature_engineering(energy)

ENCODER_PATH = root_dir / "models" / "month_encoder.pkl"

encoder = joblib.load(ENCODER_PATH)

# Separate feature from target and preprocess predictors
X = xgb_preprocess(energy, encoder)
y = energy['Appliances']


# Train-calibration-test split (70-15-15)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, shuffle= False
)

X_cal, X_test, y_cal, y_test = train_test_split(
    X_test, y_test, shuffle= False
)

## **Import Model**

In [ ]:
MODEL_PATH = PROJECT_ROOT/"models"/"xgb_best.pkl"

# Use joblib.load instead of pickle.load
model = joblib.load(MODEL_PATH)

## **Apply Adaptive Conformal Prediction**

In [ ]:
# Predict calibration set
y_pred_cal = np.asarray(model.predict(X_cal))
y_pred_cal = pd.DataFrame(y_pred_cal)

In [ ]:
# Predict test set
y_pred_test = np.asarray(model.predict(X_test))
y_pred_test = pd.DataFrame(y_pred_test)

In [ ]:
def conformal_quantile(scores, alpha):
    scores = np.asarray(scores, dtype=float)

    if scores.size == 0:
        raise ValueError("Calibration scores cannot be empty.")

    if not np.all(np.isfinite(scores)):
        raise ValueError("Calibration scores must be finite.")

    if not 0.0 <= alpha <= 1.0:
        raise ValueError("alpha must be between 0 and 1.")

    if alpha == 1.0:
        return 0.0

    n = len(scores)
    k = int(np.ceil((n + 1) * (1.0 - alpha)))

    if k > n:
        return np.inf

    return float(np.partition(scores, k - 1)[k - 1])

In [ ]:
# 1. Calculate calibration residuals
cal_residuals = np.abs(y_cal - y_pred_cal)

if len(cal_residuals) == 0:
    raise ValueError("Calibration set cannot be empty.")

# 2. Initialize ACI
tracker = ACITracker(alpha=0.2, gamma=0.01)

prediction_intervals = []
coverage_results = []
alpha_values = []

# 3. Online prediction loop
for t in range(len(y_test)):
    y_true = float(y_test[t])
    pred = float(y_pred_test[t])

    # Record the alpha used for this prediction
    current_alpha = tracker.alpha_t
    alpha_values.append(current_alpha)

    # Calculate the conformal margin
    effective_alpha = float(
        np.clip(current_alpha, 0.0, 1.0)
    )

    margin = conformal_quantile(
        cal_residuals,
        effective_alpha
    )

    # Construct the prediction interval
    lower_bound = pred - margin
    upper_bound = pred + margin

    # Apply only if the target is nonnegative
    lower_bound = max(0.0, lower_bound)

    prediction_intervals.append(
        (lower_bound, upper_bound)
    )

    # Evaluate coverage
    is_covered = (
        lower_bound <= y_true <= upper_bound
    )

    coverage_results.append(is_covered)

    # Update alpha for the NEXT prediction
    err_t = 0.0 if is_covered else 1.0
    tracker.update(err_t)

# 4. Evaluate results
coverage_results = np.asarray(coverage_results, dtype=bool)
prediction_intervals = np.asarray(
    prediction_intervals, dtype=float
)

empirical_coverage = coverage_results.mean()

interval_widths = (
    prediction_intervals[:, 1]
    - prediction_intervals[:, 0]
)

mean_interval_width = interval_widths.mean()

print(f"Target coverage: {1 - tracker.alpha:.1%}")
print(f"Empirical coverage: {empirical_coverage:.1%}")
print(f"Mean interval width: {mean_interval_width:.4f}")

In [ ]:
lb = [interval[0] for interval in prediction_intervals]
ub = [interval[1] for interval in prediction_intervals]

In [ ]:
plt.plot(np.asarray(ub) - np.asarray(lb))

In [ ]:
# Unpack prediction intervals into 1D upper and lower bounds
intervals = np.array(prediction_intervals)
lower_bounds = intervals[:, 0].squeeze()
upper_bounds = intervals[:, 1].squeeze()

# 3. Create time steps based on the TEST set length (not y_cal)
time_steps = np.arange(len(y_test))

# 4. Create plot figure
plt.figure(figsize=(14, 6))

# Plot actual ground truth values
plt.plot(time_steps, y_test, label="Actual (y_true)", color="#0284c7", linewidth=1.5, alpha = 0.5)

# Plot model point predictions
plt.plot(
    time_steps,
    y_pred_test,
    label="Predicted Consumption",
    color='black',
    linewidth=1.2,
)


# Shade the prediction interval
plt.fill_between(
    time_steps,
    lb,  # or lb
    ub,  # or ub
    color="red",
    alpha=0.15,  
    label="Conformal Interval Prediction"
)


plt.title("Adaptive Conformal Prediction Intervals", fontsize=14, pad=10)
plt.xlabel("Time Step (t)", fontsize=11)
plt.ylabel("Value", fontsize=11)
plt.legend(loc="upper left", frameon=True)
plt.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
alpha_values = np.asarray(alpha_values)

plt.figure(figsize=(12, 4))
plt.plot(alpha_values, label="Adaptive alpha")
plt.axhline(
    0.05,
    linestyle="--",
    label="Target alpha = 0.05"
)
plt.xlabel("Time Step (t)")
plt.ylabel("Miscoverage Level")
plt.title("ACI Adaptive Miscoverage Level")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()